In [12]:
# -----------------------------
# 02_train_vgg16.ipynb
# -----------------------------

# Importations
import numpy as np
import torch
import torch.nn as nn
import os
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision import models, transforms
import torch.optim as optim
import mlflow
import mlflow.pytorch
from torchvision.models import vgg16, VGG16_Weights
from mlflow.tracking import MlflowClient
from mlflow.entities import ViewType


In [3]:
# -----------------------------
# Paramètres
# -----------------------------
#le modèle voit 32 images puis calcule la perte et met à jour les poids.
#32 est un compromis classique pour les GPU moyens.
BATCH_SIZE = 32
#Nombre de passages complets sur tout le dataset(si trop d'epochs le modèle peut apprendre par cœur les images.)
EPOCHS = 10
#Détermine la taille du pas pour la mise à jour des poids à chaque backpropagation.
#(On ne veut pas changer brutalement les poids → learning rate petit.)
LEARNING_RATE = 1e-4
#Permet de passer les calculs sur GPU si disponible, sinon CPU (Accélération calcul)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224  # Doit correspondre au preprocessing

In [4]:
# -----------------------------
# Chargement des données
# -----------------------------
# Si grayscale → 1 canal
# Transforme en tenseur PyTorch et ajoute dimension canal
# Chemin vers le dossier processed
PROCESSED_DIR = "../data/processed"

# Charger les fichiers numpy
images = np.load(os.path.join(PROCESSED_DIR, "images.npy"))  # shape = [N,224,224,1]
labels = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)  # shape = [N,1,224,224]
y = torch.tensor(labels, dtype=torch.long)

# Créer Dataset et DataLoader
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batch shape:", next(iter(train_loader))[0].shape)
print("Validation batch shape:", next(iter(val_loader))[0].shape)



Train batch shape: torch.Size([32, 1, 224, 224, 1])
Validation batch shape: torch.Size([32, 1, 224, 224, 1])


In [5]:
# -----------------------------
# Définition du modèle VGG16
# -----------------------------
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

# Adapter la première couche pour 1 canal
old_conv = model.features[0]
new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding
)
with torch.no_grad():
    new_conv.weight[:, 0, :, :] = old_conv.weight.mean(dim=1)
model.features[0] = new_conv

# Modifier la dernière couche pour 2 classes
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 2)
model = model.to(DEVICE)


In [6]:
# Loss et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [7]:
# -----------------------------
# Configuration MLflow
# -----------------------------
mlflow.set_tracking_uri("file:./mlruns")  # dossier mlruns dans le projet
mlflow.set_experiment("VGG16_model")


2025/12/16 12:13:07 INFO mlflow.tracking.fluent: Experiment with name 'VGG16_model' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/USER/Desktop/ecg-classification/notebooks/mlruns/625405542091422125', creation_time=1765883587939, experiment_id='625405542091422125', last_update_time=1765883587939, lifecycle_stage='active', name='VGG16_model', tags={}>

In [ ]:
# Vérifie si un run fantôme existe
if mlflow.active_run() is not None:
    print("Un run fantôme détecté :", mlflow.active_run().info.run_id)
    
    # Solution compatible MLflow 2.x / 3.x
    mlflow.active_run()._run_id = None  # réinitialise le run actif
    print("Run fantôme réinitialisé ✅")


In [9]:
# -----------------------------
# Entraînement avec MLflow
# -----------------------------
with mlflow.start_run(run_name="VGG16_Run") as run:

    mlflow.log_param("model", "VGG16")
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("optimizer", "Adam")

    for epoch in range(EPOCHS):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            if inputs.dim() == 5 and inputs.size(-1) == 1:
                    inputs = inputs.squeeze(-1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

        train_loss /= train_size
        train_acc = correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                if inputs.dim() == 5 and inputs.size(-1) == 1:
                    inputs = inputs.squeeze(-1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()

        val_loss /= val_size
        val_acc = val_correct / val_total

        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)
 # ----- Sauvegarde du modèle -----
    model.eval()
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="VGG16_model",
    )

    print("Run MLflow terminé avec succès :", run.info.run_id)

Epoch [1/10] Train Loss: 0.6713 Train Acc: 0.5738 Val Loss: 0.5900 Val Acc: 0.6827
Epoch [2/10] Train Loss: 0.3438 Train Acc: 0.8596 Val Loss: 0.2189 Val Acc: 0.8846
Epoch [3/10] Train Loss: 0.1916 Train Acc: 0.9153 Val Loss: 0.1105 Val Acc: 0.9615
Epoch [4/10] Train Loss: 0.1356 Train Acc: 0.9467 Val Loss: 0.0999 Val Acc: 0.9615
Epoch [5/10] Train Loss: 0.0743 Train Acc: 0.9806 Val Loss: 0.1489 Val Acc: 0.9519
Epoch [6/10] Train Loss: 0.1147 Train Acc: 0.9709 Val Loss: 0.0618 Val Acc: 0.9904
Epoch [7/10] Train Loss: 0.0419 Train Acc: 0.9855 Val Loss: 0.1287 Val Acc: 0.9615
Epoch [8/10] Train Loss: 0.0270 Train Acc: 0.9903 Val Loss: 0.1033 Val Acc: 0.9808
Epoch [9/10] Train Loss: 0.0157 Train Acc: 0.9976 Val Loss: 0.0639 Val Acc: 0.9808
Epoch [10/10] Train Loss: 0.0120 Train Acc: 0.9952 Val Loss: 0.0603 Val Acc: 0.9904
Run MLflow terminé avec succès : 871a52a5f1944f7c9a15aded637dc69c
